# RETECO · listwise-реранк LLM (RankGPT / ReasonRank)

Модель видит сразу 20 кандидатов (только тексты, без `doc_id`) и выдаёт их
порядок `[3] > [1] > …`. Скользящее окно 20 / шаг 10 снизу вверх, как в RankGPT
и ReasonRank; при `TOPK = 20` это одно окно.

Модели: A100 — `liuwenhan/reasonrank-7B` (обучен ровно на это, сначала
рассуждает в `<think>`, потом `<answer>`); L4/T4 — Qwen3-4B-Instruct-2507 с
промптом RankGPT. Промпт ReasonRank взят дословно из их кода
(`listwise_prompt_r1.toml`, `add_prefix_prompt`, `add_post_prompt`).

**Как запускать.** Прогон первой стадии должен лежать на Drive в
`reteco/runs/<FIRST_STAGE>/`. Среда **A100** → «Выполнить все». Оборвалось —
ещё раз, готовые домены пропускаются.

**Сколько.** 858 реплик × 1 окно: reasonrank-7B на A100 ≈ 10–20 мин;
`TOPK = 100` (9 окон) — около часа.

**Что получится.** `reteco/runs/llmrr_<модель>_<первая стадия>_<запрос>/`.
Смешивание с первой стадией 0.6 / 0.4 — в `analysis.ipynb`.

In [ ]:
# ── 1. Настройки ─────────────────────────────────────────────────────────────
MODEL = "auto"        # A100 → liuwenhan/reasonrank-7B, иначе Qwen/Qwen3-4B-Instruct-2507
FIRST_STAGE = "bm25_fuse3_dev"   # папка в Drive/reteco/runs
QUERY_SET = "qd"      # что видит модель как запрос: "qd" — самостоятельный вопрос Claude,
                      # "hist" — реплика + история, как у организаторов; или папка в queries
SPLIT = "dev"
TOPK, WINDOW, STEP = 20, 20, 10
PASSAGE_TOKENS = 300  # документ обрезается до стольких токенов
MAXLEN = 16384
DRIVE = "/content/drive/MyDrive/reteco"

In [ ]:
# ── 2. Drive, vLLM, GPU ─────────────────────────────────────────────────────
import os, sys, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
from google.colab import drive
drive.mount("/content/drive")
!pip -q install vllm openai pytrec-eval-terrier 2>&1 | tail -2
import torch

# vllm ставит свой torch, а torchvision/torchaudio из Colab остаются под другую
# CUDA, и import vllm падает. Лечение из occ-rag-bfcl: torchaudio снести,
# torchvision поставить под CUDA нового torch. После правки нужен перезапуск.
def _imports_ok(mod):
    try:
        __import__(mod); return True
    except RuntimeError:
        return False       # стоит, но собран под другую CUDA
    except ImportError:
        return True        # не стоит — и конфликта нет
fixed = False
if not _imports_ok("torchaudio"):
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchaudio"], check=True)
    fixed = True
if not _imports_ok("torchvision") and "+" in torch.__version__:
    base, cu = torch.__version__.split("+")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "--force-reinstall", "--no-deps",
                    f"torchvision==0.{int(base.split('.')[1]) + 15}.*",
                    "--index-url", f"https://download.pytorch.org/whl/{cu}"], check=True)
    fixed = True
if fixed:
    raise RuntimeError("Зависимости исправлены. Среда выполнения → Перезапустить сеанс, "
                       "потом снова «Выполнить все».")
if globals().get("NEED_GPU", True):
    assert torch.cuda.is_available(), "Нет GPU: Среда выполнения → Сменить среду выполнения → A100 или L4"
MEM = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
DTYPE = "bfloat16" if MEM and torch.cuda.is_bf16_supported() else "half"   # T4 bf16 не умеет
BIG = MEM >= 39                                                             # A100 40/80 ГБ
print(torch.cuda.get_device_name(0) if MEM else "без GPU", f"{MEM:.0f} ГБ, {DTYPE}, torch", torch.__version__)

def load_llm(model, maxlen, util=0.88):
    """На время конструктора — настоящие stdout/stderr: в ядре Jupyter vLLM падает
    на sys.stdout.fileno() ещё до загрузки весов (occ-rag-bfcl)."""
    from vllm import LLM
    out, err = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = sys.__stdout__, sys.__stderr__
    try:
        return LLM(model=model, dtype=DTYPE, max_model_len=maxlen, gpu_memory_utilization=util, seed=0)
    finally:
        sys.stdout, sys.stderr = out, err

def chat(tok, msgs):
    """Промпт по шаблону модели; у Qwen3 размышления выключены (быстро, детерминированно)."""
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)

In [ ]:
# ── 3. Данные RETECO, трек 2 (≈270 МБ, минута) ──────────────────────────────
from huggingface_hub import snapshot_download
snapshot_download("DataScience-UIBK/RETECO-SemEval2027", repo_type="dataset",
                  local_dir="/content/reteco",
                  allow_patterns=["track2_recor/*/documents.jsonl",
                                  "track2_recor/*/benchmark_*.json",
                                  "track2_recor/*/qrels_*.txt"])

In [ ]:
# ── 4. Общий код. Повторяет scripts/retrieve.py и score.py из репозитория ──
import os, re, json, time, glob
import numpy as np, pytrec_eval
from tqdm.auto import tqdm

DATA = "/content/reteco/track2_recor"
DOMAINS = ["drones", "politics", "law", "medicalsciences", "hardware", "economics",
           "psychology", "biology", "sustainable_living", "robotics", "earth_science"]  # от малых к большим
# официальный BM25 «реплика + история», dev, nDCG@10 — для сравнения
BM25_DEV = {"biology": .6065, "drones": .2954, "earth_science": .6264, "economics": .4739,
            "hardware": .3378, "law": .3437, "medicalsciences": .1730, "politics": .3219,
            "psychology": .5726, "robotics": .5123, "sustainable_living": .5531}
BM25_MACRO = {"dev": 0.4379, "train": 0.4539}
RUNS, QUERIES = f"{DRIVE}/runs", f"{DRIVE}/queries"
os.makedirs(RUNS, exist_ok=True); os.makedirs(QUERIES, exist_ok=True)

def load_corpus(dom):
    ids, texts = [], []
    for ln in open(f"{DATA}/{dom}/documents.jsonl", encoding="utf-8"):
        if ln.strip():
            d = json.loads(ln); ids.append(d["doc_id"]); texts.append(d["content"])
    return ids, texts

def topics(dom, split, qset):
    """Реплики без золота пропускаются; история приклеивается как у организаторов.
    qset: "query", "hist" или имя папки в Drive/reteco/queries (варианты rewrite.py)."""
    given = None
    if qset not in ("query", "hist"):
        with open(f"{QUERIES}/{qset}/{split}.jsonl", encoding="utf-8") as f:
            # ключ (домен, topic_id): в доменах BRIGHT topic_id повторяются между доменами
            given = {(r["domain"], r["topic_id"]): r["text"] for r in map(json.loads, filter(str.strip, f))}
    ids, texts = [], []
    for c in json.load(open(f"{DATA}/{dom}/benchmark_{split}.json", encoding="utf-8")):
        for t in c["turns"]:
            if not (t.get("gold_doc_ids") or t.get("supporting_doc_ids")):
                continue
            qid = f"{c['id']}_turn_{t['turn_id']}"
            if given is not None:
                text = given[(dom, qid)]
            else:
                text, h = t["query"], t.get("conversation_history", "")
                if qset == "hist" and h and h != "No previous conversation.":
                    text += f"\n\nConversation History:\n{h}"
            ids.append(qid); texts.append(text)
    return ids, texts

def ndcg10(run, dom, split):
    """run: {qid: {doc: score}}. Реплика без прогона получает 0, как на лидерборде."""
    qrels = {}
    for ln in open(f"{DATA}/{dom}/qrels_{split}.txt"):
        p = ln.split()
        if len(p) == 4:
            qrels.setdefault(p[0], {})[p[2]] = int(p[3])
    res = pytrec_eval.RelevanceEvaluator(qrels, {"ndcg_cut.10"}).evaluate(run)
    return sum(v["ndcg_cut_10"] for v in res.values()) / len(qrels)

def read_run(path):
    run = {}
    for ln in open(path, encoding="utf-8"):
        p = ln.split()
        if len(p) == 6:
            run.setdefault(p[0], {})[p[2]] = float(p[4])
    return run

def write_run(path, run, tag):
    """Сначала во временный файл, потом переименование: оборванная запись не
    оставляет полуфайл, который перезапуск принял бы за готовый."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        for q, docs in run.items():
            for r, (d, s) in enumerate(sorted(docs.items(), key=lambda x: -x[1]), 1):
                f.write(f"{q}\tQ0\t{d}\t{r}\t{s:.6f}\t{tag}\n")
    os.replace(path + ".tmp", path)

In [ ]:
# ── 5. Модель и промпт (ReasonRank: rank_GPT_reasoning; прочие: rank_GPT) ────
from vllm import SamplingParams
if MODEL == "auto":
    MODEL = "liuwenhan/reasonrank-7B" if BIG else "Qwen/Qwen3-4B-Instruct-2507"
REASON = "reasonrank" in MODEL.lower()
llm = load_llm(MODEL, MAXLEN)
tok = llm.get_tokenizer()
SP = SamplingParams(temperature=0.0, max_tokens=3072 + 100 if REASON else 300)   # 3072 — reasoning_maxlen у ReasonRank
SYS = ("You are RankLLM, an intelligent assistant that can rank passages based on their relevance to the query."
       + (" Given a query and a passage list, you first thinks about the reasoning process in the mind and then "
          "provides the answer (i.e., the reranked passage list). The reasoning process and answer are enclosed "
          "within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> reasoning process "
          "here </think> <answer> answer here </answer>." if REASON else ""))
num_fix = lambda s: re.sub(r"\[(\d+)\]", r"(\1)", s)     # [n] в тексте спутал бы номера кандидатов

def passage(text):
    return num_fix(tok.convert_tokens_to_string(tok.tokenize(text.strip())[:PASSAGE_TOKENS]))

def prompt(query, docs):
    q, n = num_fix(query).strip(), len(docs)
    body = (f"I will provide you with {n} passages, each indicated by a numerical identifier []. "
            f"Rank the passages based on their relevance to the search query: {q}.\n\n")
    body += "".join(f"[{i}] {d}\n" for i, d in enumerate(docs, 1))
    body += (f"Search Query: {q}.\nRank the {n} passages above based on their relevance to the search query. "
             "All the passages should be included and listed using identifiers, in descending order of relevance. ")
    body += ("The format of the answer should be [] > [], e.g., [2] > [1]." if REASON else
             "The output format should be [] > [], e.g., [2] > [1]. Only respond with the ranking results, "
             "do not say any word or explain.")
    return chat(tok, [{"role": "system", "content": SYS}, {"role": "user", "content": body}])

def permutation(text, n):
    """Порядок 0..n-1 из ответа: сначала названные номера без повторов, потом забытые
    в исходном порядке (как _clean_response в rank_llm)."""
    m = re.search(r"<answer>(.*?)(?:</answer>|$)", text, re.S)
    body = m.group(1) if m else text.split("</think>")[-1]
    seen = []
    for x in re.findall(r"\[(\d+)\]", body) or re.findall(r"\d+", body):
        i = int(x) - 1
        if 0 <= i < n and i not in seen:
            seen.append(i)
    return seen + [i for i in range(n) if i not in seen], len(seen)
print(MODEL, "· рассуждает" if REASON else "· RankGPT")

In [ ]:
# ── 6. Реранк по доменам → Drive ────────────────────────────────────────────
src = f"{RUNS}/{FIRST_STAGE}"
assert os.path.isdir(src), f"нет прогона первой стадии {src}"
dst = f"{RUNS}/llmrr_{MODEL.split('/')[-1]}_{FIRST_STAGE}_{QUERY_SET}"
stats = {"windows": 0, "empty": 0, "partial": 0, "cut": 0}
rows = []
for dom in DOMAINS:
    path = f"{dst}/{dom}/run.trec"
    first = read_run(f"{src}/{dom}/run.trec")
    if not os.path.exists(path):
        ids, texts = load_corpus(dom)
        text = dict(zip(ids, texts))
        qids, qtexts = topics(dom, SPLIT, QUERY_SET)
        qtext = dict(zip(qids, qtexts))
        order = {q: [d for d, _ in sorted(first.get(q, {}).items(), key=lambda x: -x[1])] for q in qids}
        t0 = time.time()
        end = min(TOPK, max(len(v) for v in order.values()))
        start = end - WINDOW
        while end > 0 and start + STEP != 0:        # окна снизу вверх, как sliding_windows в rank_llm
            start = max(start, 0)
            live = [q for q in qids if len(order[q][start:end]) > 1]
            outs = llm.generate([prompt(qtext[q], [passage(text[d]) for d in order[q][start:end]]) for q in live],
                                SP, use_tqdm=False)
            for q, o in zip(live, outs):
                win = order[q][start:end]
                perm, named = permutation(o.outputs[0].text, len(win))
                order[q][start:end] = [win[i] for i in perm]
                stats["windows"] += 1
                stats["empty"] += named == 0
                stats["partial"] += 0 < named < len(win)
                stats["cut"] += o.outputs[0].finish_reason == "length"
            end, start = end - STEP, start - STEP
        run = {q: {d: (len(v) - r) / len(v) for r, d in enumerate(v)} for q, v in order.items() if v}
        write_run(path, run, "llmrr")
        print(f"{dom}: {len(qids)} реплик за {(time.time() - t0) / 60:.1f} мин")
    a, b = ndcg10(first, dom, SPLIT), ndcg10(read_run(path), dom, SPLIT)
    rows.append((a, b))
    print(f"  {dom:<20} первая стадия {a:.4f} → LLM-реранк {b:.4f}  ({b - a:+.4f})")
json.dump({"model": MODEL, "first_stage": FIRST_STAGE, "query_set": QUERY_SET, "split": SPLIT, "topk": TOPK,
           "window": WINDOW, "step": STEP, "passage_tokens": PASSAGE_TOKENS, "prompt": "rank_GPT_reasoning" if REASON
           else "rank_GPT", "stats": stats, "gpu": torch.cuda.get_device_name(0),
           "date": time.strftime("%Y-%m-%dT%H:%M:%S")}, open(f"{dst}/config.json", "w"), ensure_ascii=False, indent=2)
A, B = (sum(r[i] for r in rows) / len(rows) for i in (0, 1))
print(f"\nМАКРО  первая стадия {A:.4f} → LLM-реранк {B:.4f}  ({B - A:+.4f})")
print("окон:", stats["windows"], "· без разбора:", stats["empty"], "· неполных:", stats["partial"],
      "· обрезано по длине:", stats["cut"], "(эти числа — только за этот запуск)")